# 04 - Graph Write and Read

Demonstrates the core document-graph primitives:
- `Node` / `Edge` models
- `CypherBuilder` — generates openCypher from models
- `DocumentGraphQueryEngine` — queries typed nodes via GraphStore

In [ ]:
import os
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from document_graph import Node, Edge
from document_graph.graph_build import node_to_cypher, edge_to_cypher
from document_graph.query import DocumentGraphQueryEngine

GRAPH_STORE = os.environ.get("GRAPH_STORE", "neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182")
print("Imports OK")


## Write Nodes

In [ ]:
graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
TENANT = "notebook_demo"

# Create typed nodes
nodes = [
    Node(id="acc-1", labels=["Account"], properties={"name": "Production", "type": "PROD", "region": "us-east-1"}),
    Node(id="acc-2", labels=["Account"], properties={"name": "Development", "type": "DEV", "region": "us-west-2"}),
    Node(id="ps-1", labels=["PermissionSet"], properties={"name": "AdminAccess", "risk": "HIGH"}),
    Node(id="ps-2", labels=["PermissionSet"], properties={"name": "ReadOnly", "risk": "LOW"}),
]

for n in nodes:
    cypher, params = node_to_cypher(n, tenant_id=TENANT)
    graph_store.execute_query(cypher, params)

print(f"✅ Wrote {len(nodes)} nodes")


## Write Edges

In [ ]:
edges = [
    Edge(id="e1", source_id="ps-1", target_id="acc-1", label="ASSIGNED_TO"),
    Edge(id="e2", source_id="ps-2", target_id="acc-1", label="ASSIGNED_TO"),
    Edge(id="e3", source_id="ps-2", target_id="acc-2", label="ASSIGNED_TO"),
]

for e in edges:
    cypher, params = edge_to_cypher(e, tenant_id=TENANT)
    graph_store.execute_query(cypher, params)

print(f"✅ Wrote {len(edges)} edges")


## Query

In [ ]:
engine = DocumentGraphQueryEngine(graph_store, tenant_id=TENANT)

print('All Accounts:')
for r in engine.get_nodes('Account'):
    print(f'  {r}')

print('\nHigh-risk Permission Sets:')
for r in engine.find_by_property('PermissionSet', 'risk', 'HIGH'):
    print(f'  {r}')

print('\nAssignments:')
for r in engine.get_relationships('PermissionSet', 'ASSIGNED_TO', 'Account'):
    print(f'  {r}')